In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)
plt.close('all')

# Load Data

In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)
NWindows = len(flowStatsMI[flowStatsMI['openingType'] == 'xwindow'])
NWindows += len(flowStatsMI[flowStatsMI['openingType'] == 'zwindow'])
display(f"N Windows: {NWindows}")
display(f"N Skylight: {len(flowStatsMI[flowStatsMI['openingType'] == 'skylight'])}")
display(f"N Rooms: {len(roomVentilationMI)}")

In [ ]:
y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
df = flowStatsMI.copy()
data = df.copy()

In [ ]:
# Use the plotting function with turbulence model
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Ventilation Model",
    p0=[1.0, 0.00000001],
    bounds=([0.99999, 0], [1.00001, 0.0000001]),
    adjustData=False,
    show_assymptotes=True
)

In [ ]:
# Use the plotting function with turbulence model
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Ventilation Model",
    p0=[1.0, 0.00000001],
    bounds=([0.1, 0], [np.inf, 0.0000001]),
    adjustData=False,
    show_assymptotes=True
)

In [ ]:
# Use the plotting function with turbulence model
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Ventilation Model",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True
)

In [ ]:
x_var2 = "rms-mass_flux-Norm"
q_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2].mean().to_dict()
q_rms_global = data[x_var2].mean()
print(f"Global mean {x_var2}: {q_rms_global:.4f}")
for sl in [0, 1]:
    for s in [1, -1]:
        label = "Skylight" if sl == 1 else "Window"
        direction = "In" if s == 1 else "Out"
        print(f"{label}-{direction} mean {x_var2}: {q_rms_means.get((sl, s), np.nan):.4f}")

# Use subgroup-specific initialization for the second fit parameter
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="Ventilation Model",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
 )

In [ ]:
def ventilationReDecomp_qMeasured(X, a, u_rms):
    u_model, rms_scaling = X
    u_rms *= rms_scaling
    return pyafn.ventilationReDecomp_q(u_model, a, u_rms)

plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2,
    hue=x_var2,
    style="slAll",
    model_func=ventilationReDecomp_qMeasured,
    model_name="Turbulence-Enhanced q",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=False,
    show_assymptotes=False,
    add_numeric_colorbar=True
 )

In [ ]:
# Use the plotting function with turbulence model
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="Ventilation Model",
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True
)

In [ ]:
x_var2 = "p_rms-noInt-Norm"
p_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2].mean().to_dict()
p_rms_global = data[x_var2].mean()
print(f"Global mean {x_var2}: {p_rms_global:.4f}")
for sl in [0, 1]:
    for s in [1, -1]:
        label = "Skylight" if sl == 1 else "Window"
        direction = "In" if s == 1 else "Out"
        print(f"{label}-{direction} mean {x_var2}: {p_rms_means.get((sl, s), np.nan):.4f}")

# Use subgroup-specific initialization for the second fit parameter
fig, axs = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="Ventilation Model",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
 )

In [ ]:
def ventilationReDecomp_pMeasured(X, a, p_rms):
    u_model, rms_scaling = X
    p_rms *= rms_scaling
    return pyafn.ventilationReDecomp_p(u_model, a, p_rms)

# Combined paper-ready q+p figure (rows: q fit, q measured, p fit, p measured; cols: Window/Skylight)
fig_p, axs_p = plt.subplots(4, 2, figsize=(13, 22), dpi=160, sharex=True, sharey=True)

# Row 1: baseline q model (categorical hue)
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$I(q)$ Fit",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    axes=axs_p[0, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Row 2: subgroup-mean initialized q model (categorical hue)
x_var2_q = "rms-mass_flux-Norm"
q_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2_q].mean().to_dict()
q_rms_global = data[x_var2_q].mean()
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$I(q)$ Bulk Measurement",
    p0=[1.0, q_rms_global],
    bounds=([0.1, q_rms_global-0.00001], [np.inf, q_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=q_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_p[1, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Row 3: baseline p model (categorical hue)
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$I(p)$ Fit",
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=False,
    show_assymptotes=True,
    axes=axs_p[2, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Row 4: subgroup-mean initialized p model (categorical hue)
x_var2_p = "p_rms-noInt-Norm"
p_rms_means = data.groupby(["skylight", "Sdelp"])[x_var2_p].mean().to_dict()
p_rms_global = data[x_var2_p].mean()
plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name="$I(p)$ Bulk Measurement",
    p0=[1.0, p_rms_global],
    bounds=([0.1, p_rms_global-0.00001], [np.inf, p_rms_global+0.00001]),
    adjustData=False,
    show_assymptotes=True,
    group_param2_means=p_rms_means,
    group_param2_half_width=0.00001,
    axes=axs_p[3, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=False,
    figure_suptitle=False,
 )

# Paper-ready labels and layout
col_titles_p = ["Window", "Skylight"]
for j, t in enumerate(col_titles_p):
    axs_p[0, j].set_title(t, fontsize=24)

row_labels_p = ["$I(q)$ Fit", "$I(q)$ Measured", "$I(p)$ Fit", "$I(p)$ Measured"]
for i, row_label in enumerate(row_labels_p):
    axs_p[i, 0].set_ylabel(row_label, fontsize=24)

for ax in axs_p.flatten():
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

fig_p.supxlabel("Modeled Flux Velocity", fontsize=24, y=0.04)
fig_p.supylabel("LES Flux Velocity", fontsize=24)
# fig_p.suptitle("Ventilation Model Fits (q and p): Window vs Skylight", fontsize=24)
fig_p.subplots_adjust(left=0.13, right=0.93, top=0.94, bottom=0.08, wspace=0.13, hspace=0.12)

In [ ]:
data["Cp"] = data["p_rms-noInt-Norm"] / (rho / 2)

# Combined point-wise figure (rows: q and p, cols: Window/Skylight)
fig_pointwise, axs_pointwise = plt.subplots(2, 2, figsize=(13, 12), dpi=160, sharex=True, sharey=True)

# Point-wise q (top row)
q_hue_norm = plt.Normalize(vmin=0, vmax=0.25)
data_q_pointwise = data.copy()
plot_ventilation_model_fit(
    data=data_q_pointwise,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_q,
    hue=x_var2_q,
    style="slAll",
    model_func=ventilationReDecomp_qMeasured,
    model_name="$I(q)$ Point-Wise Measurement",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=True,
    axes=axs_pointwise[0, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=q_hue_norm,
    palette_numeric="plasma",
    figure_suptitle=False,
    colorbar_rect=[0.92, 0.56, 0.015, 0.30],
    colorbar_orientation="vertical",
)

# Point-wise p (bottom row)
x_var2_p = "p_rms-noInt-Norm"
p_hue_norm = plt.Normalize(vmin=0, vmax=0.25)

data_p_pointwise = data.copy()
plot_ventilation_model_fit(
    data=data_p_pointwise,
    y_var=y_var,
    x_var=x_var,
    x_var2=x_var2_p,
    hue="Cp",
    style="slAll",
    model_func=ventilationReDecomp_pMeasured,
    model_name="$I(p)$ Point-Wise Measurement",
    p0=[1.0, 1.0],
    bounds=([0.1, .9999], [np.inf, 1.00001]),
    adjustData=True,
    show_assymptotes=False,
    axes=axs_pointwise[1, :],
    show_row_titles=False,
    set_axis_labels=False,
    show_figure_legend=False,
    add_numeric_colorbar=True,
    hue_norm=p_hue_norm,
    palette_numeric="viridis",
    figure_suptitle=False,
    colorbar_rect=[0.92, 0.14, 0.015, 0.30],
    colorbar_orientation="vertical",
)

# Labels and layout
col_titles_pointwise = ["Window", "Skylight"]
for j, t in enumerate(col_titles_pointwise):
    axs_pointwise[0, j].set_title(t, fontsize=24)

axs_pointwise[0, 0].set_ylabel("$I(q)$", fontsize=24)
axs_pointwise[1, 0].set_ylabel("$I(p)$", fontsize=24)

for ax in axs_pointwise.flatten():
    ax.set_xlabel("")
    ax.tick_params(labelsize=16)

fig_pointwise.supxlabel("Fluctuation-Corrected Modeled Flux Velocity", fontsize=24, y=0.05)
fig_pointwise.supylabel("LES Flux Velocity", fontsize=24)
# fig_pointwise.suptitle("Point-Wise Ventilation Model Fits", fontsize=24)
fig_pointwise.subplots_adjust(left=0.10, right=0.90, top=0.90, bottom=0.12, wspace=0.2, hspace=0.1)

In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=data,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$I(q)$ Fit",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

In [ ]:
plt.figure()
plt.scatter(xAdjusted, data[y_var], alpha=0.5, s=1)

In [ ]:
def aggregate_window_series_to_room(roomVentilationMI, window_series, out_col="xAdjusted_room", mode="abs_half_sum"):
    """
    window_series: pd.Series indexed like (run, windowKey), e.g. xAdjusted
    mode:
      - "abs_half_sum": sum(abs(q_i))/2  (ventilation magnitude style)
      - "sum":          signed sum(q_i)
      - "mean":         arithmetic mean
    """
    rv = roomVentilationMI.copy()
    rv[out_col] = np.nan

    window_key_cols = rv.columns[rv.columns.str.contains("windowKeys")].tolist()

    for (run, room), row in rv.iterrows():
        keys = row[window_key_cols].dropna().tolist()
        vals = []
        for k in keys:
            idx = (run, k)
            if idx in window_series.index:
                vals.append(window_series.loc[idx])

        if len(vals) == 0:
            continue

        vals = np.asarray(vals, dtype=float)
        if mode == "abs_half_sum":
            rv.loc[(run, room), out_col] = np.nansum(np.abs(vals)) / 2.0
        elif mode == "sum":
            rv.loc[(run, room), out_col] = np.nansum(vals)
        elif mode == "mean":
            rv.loc[(run, room), out_col] = np.nanmean(vals)
        else:
            raise ValueError(f"Unknown mode: {mode}")

    return rv

In [ ]:
minVent = roomVentilationMI[x_var].min()
maxVent = roomVentilationMI[x_var].max()
x_space = np.linspace(minVent, maxVent, 100)

windowModels = {}
for key, params in fittedParams.items():
    windowModels[key] = pyafn.ventilationReDecomp_q(x_space, *params)

roomModels = {}
roomResiduals = {}
for inOpening in ["Window", "Skylight"]:
    for outOpening in ["Window", "Skylight"]:
        roomModels[f"In {inOpening}; Out {outOpening}"] = (windowModels[(inOpening, 'Flow Entering')] + windowModels[(outOpening, 'Flow Exiting')]) / 2
        roomResiduals[f"In {inOpening}; Out {outOpening}"] = windowModels[(inOpening, 'Flow Entering')] - windowModels[(outOpening, 'Flow Exiting')]


In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI_adj, xAdjusted, out_col="xAdjusted_room_residual", mode="mean"
)
roomVentilationMI_adj["xAdjusted_room_residual"] *= 2
roomVentilationMI_adj["xAdjusted_diff"] = roomVentilationMI_adj["xAdjusted_room"] - roomVentilationMI_adj[x_var]

plot_df = roomVentilationMI_adj.copy().sort_values("roomType")

fig, axs = plt.subplots(2, 2, figsize=(13, 11), dpi=160)

for i in range(2):
    ax = axs[0, i]
    ax.plot(x_space, x_space, 'r--', label='1:1 Line')

styles = ['-', '--', '-.', ':']
for i, (key, model) in enumerate(roomModels.items()):
    color = "k"#plt.get_cmap("tab20")(i % 20)
    axs[1, 0].plot(x_space, model, linestyle=styles[i % len(styles)], label=key, color=color, alpha=1, linewidth = 1)
    axs[1, 0].legend(fontsize=16)

for i, (key, residual) in enumerate(roomResiduals.items()):
    color = "k" #plt.get_cmap("tab20")(i % 20)
    axs[1, 1].plot(
        roomModels[key] - x_space,
        residual,
        linestyle=styles[i % len(styles)],
        label=key,
        color=color,
        alpha=1,
        linewidth = 1
    )
    axs[1, 1].set_xlim(-0.1, 0.1)
    axs[1, 1].set_ylim(-0.1, 0.1)

scatter_specs = [
    (x_var, "flux-Norm", "Baseline Model vs LES"),
    ("xAdjusted_room", "flux-Norm", "Aggregated Adjusted Model vs LES"),
    (x_var, "xAdjusted_room", "Baseline Model vs Aggregated Adjusted Model"),
    ("xAdjusted_diff", "xAdjusted_room_residual", "Aggregation Difference vs Residual"),
]

for ax, (xcol, ycol, title) in zip(axs.flatten(), scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=ycol,
        hue="roomType",
        style = "slAll",
        palette="colorblind",
        alpha=0.6,
        s=10,
        ax=ax,
        legend=ax is axs[0, 0],
    )
 
    ax.set_title(title, fontsize=20)
    ax.set_xlabel(xcol, fontsize=18)
    ax.set_ylabel(ycol, fontsize=18)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=16)

handles, labels = axs[0, 0].get_legend_handles_labels()

for ax in axs.flatten():
    if ax.get_legend() is not None:
        ax.get_legend().remove()

fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(0.90, 0.5),
    fontsize=18,
    title="Room Type",
    title_fontsize=18,
    frameon=False,
)

fig.suptitle("Room-Level Aggregated Ventilation Comparisons", fontsize=24)
fig.subplots_adjust(left=0.10, right=0.87, top=0.90, bottom=0.10, wspace=0.28, hspace=0.30)

## ASHRAE Ventilation Rates

In [ ]:
windowASHRAE = pd.read_csv(f"{home_dir}/windowASHRAE.csv")

ashrae_lookup = windowASHRAE.set_index(["windowType", "roomA", "C"])["ventilationRate"]
data["ASHRAE"] = data.apply(
    lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plot_df = data[data["slAll"] == False].copy()

fig, axs = plt.subplots(1, 3, figsize=(12, 4.8), dpi=160, sharey=True, layout="constrained")

scatter_specs = [
    ("ASHRAE", "ASHRAE Pressures",  "Modeled Flux Velocity"),
    (x_var, "LES Pressures", "Modeled Flux Velocity"),
    (xAdjusted, "LES Pressures", "Fluctuation-Adjusted Flux Velocity"),
]

for ax, (xcol, title, xlabel) in zip(axs, scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=y_var,
        hue="roomType",
        # style="roomType",
        palette="colorblind",
        alpha=0.65,
        s=20,
        ax=ax,
        legend=ax is axs[0],
    )

    lim_min = np.nanmin([plot_df[x_var].min(), plot_df[y_var].min()])
    lim_max = np.nanmax([plot_df[x_var].max(), plot_df[y_var].max()])
    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "r--",
        linewidth=1.2,
        alpha=0.7,
        label="1:1" if ax is axs[0] else None,
    )

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("LES Flux Velocity", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=14)
    # ax.set_xlim(-0.6, 0.6)
    # ax.set_ylim(-0.6, 0.6)


handles, labels = axs[0].get_legend_handles_labels()
for ax in axs:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# fig.legend(
#     handles,
#     labels,
#     loc="center left",
#     bbox_to_anchor=(0.90, 0.5),
#     fontsize=12,
#     # title="Room A / Window Type",
#     title_fontsize=13,
#     frameon=False,
# )

fig.suptitle("Window-Level Ventilation Comparison", fontsize=20)
fig.subplots_adjust(left=0.08, right=0.86, top=0.83, bottom=0.18, wspace=0.30)

In [ ]:
data_WA_mean = data[data["slAll"] == False].groupby(["windowType", "roomA"])[y_var].mean().reset_index()

windowTypeOrder = data_WA_mean["windowType"].dropna().unique()
plt.figure()
sns.lineplot(data=data_WA_mean, x="roomA", y=y_var, hue="windowType", palette="tab10", hue_order=windowTypeOrder)
sns.lineplot(data=windowASHRAE, x="roomA", y="ventilationRate", hue="windowType", palette="tab10", linestyle='--', hue_order=windowTypeOrder)

In [ ]:
roomASHRAE = pd.read_csv(f"{home_dir}/roomASHRAE.csv")

ashrae_lookup = roomASHRAE.set_index(["roomType", "AofA", "C"])["ventilationRate"]
roomVentilationMI_adj["ASHRAE"] = roomVentilationMI_adj.apply(
    lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plt.figure()
sns.scatterplot(data=roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False], x="ASHRAE", y=y_var, hue="AofA", palette="viridis")

In [ ]:
room_WA_mean = roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False].groupby(["roomType", "AofA"])[y_var].mean().reset_index()

roomTypeOrder = room_WA_mean["roomType"].dropna().unique()
plt.figure()
sns.lineplot(data=room_WA_mean, x="AofA", y=y_var, hue="roomType", palette="tab10", hue_order=roomTypeOrder)
sns.lineplot(data=roomASHRAE, x="AofA", y="ventilationRate", hue="roomType", palette="tab10", linestyle='--', hue_order=roomTypeOrder)

In [ ]:
def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
import flowEmulationUtils as feUtils
flowStatsMI_directNetwork, roomVentilationMI_directNetwork = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
def _comparison_metrics(frame, x_spec, y_col):
    if isinstance(x_spec, str):
        x = frame[x_spec]
        x_label = x_spec
    else:
        x = pd.Series(x_spec).reindex(frame.index)
        x_label = getattr(x_spec, "name", None) or "x_adjusted"

    valid = pd.DataFrame({"x": x, "y": frame[y_col]}).dropna()
    residual = valid["x"] - valid["y"]

    return {
        "x_label": x_label,
        "rmse": np.sqrt(np.mean(residual ** 2)),
        "bias": residual.mean(),
        "std": residual.std(ddof=0),
        "n": len(valid),
    }


def _comparison_row(frame, x_spec, y_col, panel, split):
    metrics = _comparison_metrics(frame, x_spec, y_col)
    if metrics["n"] == 0:
        return None

    return {
        "panel": panel,
        "split": split,
        "rmse": metrics["rmse"],
        "bias": metrics["bias"],
        "std": metrics["std"],
        "n": metrics["n"],
        "x_label": metrics["x_label"],
    }


def _flow_subgroup_masks(frame):
    if "Sdelp" not in frame.columns:
        return []

    if "openingType" in frame.columns:
        opening_group = pd.Series(
            np.where(
                frame["openingType"].astype(str).str.contains("skylight", case=False, na=False),
                "Skylight",
                "Window",
            ),
            index=frame.index,
        )
    elif "skylight" in frame.columns:
        opening_group = pd.Series(
            np.where(frame["skylight"].fillna(0).astype(int) == 1, "Skylight", "Window"),
            index=frame.index,
        )
    else:
        return []

    direction_group = pd.Series(
        np.where(
            frame["Sdelp"] > 0,
            "Flow Entering",
            np.where(frame["Sdelp"] < 0, "Flow Exiting", np.nan),
        ),
        index=frame.index,
    )

    masks = []
    for opening_label in ["Window", "Skylight"]:
        for direction_label in ["Flow Entering", "Flow Exiting"]:
            mask = (opening_group == opening_label) & (direction_group == direction_label)
            if mask.any():
                masks.append((f"{opening_label}, {direction_label}", mask))

    return masks


def _panel_plot_data(frame, x_spec):
    plot_frame = frame.copy()
    if isinstance(x_spec, str):
        return plot_frame, x_spec

    x_label = getattr(x_spec, "name", None) or "x_adjusted"
    plot_frame[x_label] = pd.Series(x_spec).reindex(plot_frame.index)
    return plot_frame, x_label


def plot_direct_network_comparison(run_label, flow_frame, room_frame):
    flow_plot = flow_frame.copy()
    room_plot = room_frame.copy()

    flow_plot["p-noInt-direct_optp0-q_model-Norm"] = (
        flow_plot["p-noInt-direct_optp0-q_model"] / flow_plot["WS"]
    )
    room_plot["p-noInt-direct_optp0-q_model-Norm"] = (
        room_plot["p-noInt-direct_optp0-q_model"] / room_plot["WS"]
    )
    flow_plot["flux-Norm"] = flow_plot["flux"] / flow_plot["WS"]

    hue_order = ["corner", "cross", "dual", "single"]
    opening_style_order = ["xwindow", "zwindow", "skylight"]

    panel_specs = [
        {
            "title": "Window Baseline vs LES",
            "frame": data,
            "x": x_var,
            "y": y_var,
            "hue": "roomType",
            "style": "openingType",
            "style_order": opening_style_order,
        },
        {
            "title": "Window Adjusted vs LES",
            "frame": flow_plot,
            "x": xAdjusted,
            "y": y_var,
            "hue": "roomType",
            "style": "openingType",
            "style_order": opening_style_order,
        },
        {
            "title": "Window Direct Network vs LES",
            "frame": flow_plot,
            "x": "p-noInt-direct_optp0-q_model-Norm",
            "y": y_var,
            "hue": "roomType",
            "style": "openingType",
            "style_order": opening_style_order,
        },
        {
            "title": "Room Baseline vs LES",
            "frame": roomVentilationMI,
            "x": x_var,
            "y": y_var,
            "hue": "roomType",
            "style": "roomType",
            "style_order": hue_order,
        },
        {
            "title": "Room Adjusted vs LES",
            "frame": roomVentilationMI_adj,
            "x": "xAdjusted_room",
            "y": y_var,
            "hue": "roomType",
            "style": "roomType",
            "style_order": hue_order,
        },
        {
            "title": "Room Direct Network vs LES",
            "frame": room_plot,
            "x": "p-noInt-direct_optp0-q_model-Norm",
            "y": y_var,
            "hue": "roomType",
            "style": "roomType",
            "style_order": hue_order,
        },
    ]

    fig, axs = plt.subplots(2, 3, figsize=(14, 9), dpi=160)
    stats_rows = []

    for ax, spec in zip(axs.flatten(), panel_specs):
        plot_frame, x_col = _panel_plot_data(spec["frame"], spec["x"])
        sns.scatterplot(
            data=plot_frame,
            x=x_col,
            y=spec["y"],
            hue=spec["hue"],
            style=spec["style"],
            palette="colorblind",
            ax=ax,
            alpha=0.7,
            s=20,
            hue_order=hue_order,
            style_order=spec["style_order"],
            legend=ax is axs[0, 0],
        )

        metrics = _comparison_metrics(spec["frame"], spec["x"], spec["y"])
        total_row = _comparison_row(spec["frame"], spec["x"], spec["y"], spec["title"], "Total")
        if total_row is not None:
            stats_rows.append(total_row)

        for split_label, mask in _flow_subgroup_masks(spec["frame"]):
            split_row = _comparison_row(
                spec["frame"].loc[mask],
                spec["x"],
                spec["y"],
                spec["title"],
                split_label,
            )
            if split_row is not None:
                stats_rows.append(split_row)

        valid = plot_frame[[x_col, spec["y"]]].dropna()
        if not valid.empty:
            lim_min = valid.min().min()
            lim_max = valid.max().max()
            ax.plot([lim_min, lim_max], [lim_min, lim_max], "r--", linewidth=1.0, alpha=0.7)
            ax.set_xlim(lim_min, lim_max)
            ax.set_ylim(lim_min, lim_max)

        ax.set_title(spec["title"], fontsize=13)
        ax.set_xlabel(metrics["x_label"], fontsize=11)
        ax.set_ylabel(spec["y"], fontsize=11)
        ax.grid(True, alpha=0.3)

    handles, labels = axs[0, 0].get_legend_handles_labels()
    for ax in axs.flatten():
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    fig.legend(
        handles,
        labels,
        loc="center left",
        bbox_to_anchor=(0.92, 0.5),
        frameon=False,
    )
    fig.suptitle(f"{run_label} Comparison", fontsize=18)
    fig.subplots_adjust(right=0.9, hspace=0.3, wspace=0.25)

    stats_df = pd.DataFrame(stats_rows)
    print(run_label)
    overall_stats = stats_df[stats_df["split"] == "Total"].drop(columns=["split", "x_label"])
    print("Overall")
    print(overall_stats.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    subgroup_stats = stats_df[stats_df["split"] != "Total"].drop(columns=["x_label"])
    if not subgroup_stats.empty:
        print("\nBy opening / flow direction")
        print(subgroup_stats.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    return fig, axs, stats_df


fig, axs, stats_direct_network = plot_direct_network_comparison(
    "Direct Network",
    flowStatsMI_directNetwork,
    roomVentilationMI_directNetwork,
)


In [ ]:
optParams = [6.121e-01, 6.467e-01, 5.840e-01, 6.464e-01, 1.271e-01, 1.010e-01, 9.888e-09, 1.309e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, optParams[4]]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, optParams[5]]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, optParams[6]]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, optParams[7]])
    }

def _params_for_opening(opening_type):
    group = "Skylight" if "skylight" in str(opening_type).lower() else "Window"
    return (
        [fittedParams[(group, "Flow Entering")][0]*Cd, fittedParams[(group, "Flow Exiting")][0]*Cd],
        [fittedParams[(group, "Flow Entering")][1], fittedParams[(group, "Flow Exiting")][1]],
    )

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI_directNetwork2, roomVentilationMI_directNetwork2 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network2 = plot_direct_network_comparison(
    "Direct Network 2",
    flowStatsMI_directNetwork2,
    roomVentilationMI_directNetwork2,
)


In [ ]:
optParams = [6.882e-01, 7.107e-01, 6.848e-01, 7.639e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, 0]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, 0]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, 0]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, 0])
    }

flowStatsMI[["C_d", "pRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

flowStatsMI["pRMS"] = flowStatsMI.apply(lambda x: [x["p_rms-noInt"], x["p_rms-noInt"]], axis=1)

flowStatsMI[["C_d", "pRMS"]]

In [ ]:
flowStatsMI_directNetwork3, roomVentilationMI_directNetwork3 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network3 = plot_direct_network_comparison(
    "Direct Network 3",
    flowStatsMI_directNetwork3,
    roomVentilationMI_directNetwork3,
)


In [ ]:
optParams = [7.022e-01, 7.195e-01, 6.934e-01, 7.408e-01]

fittedParams = {
    ('Window',   'Flow Entering'): np.array([optParams[0]/Cd, 0]),
    ('Window',   'Flow Exiting'):  np.array([optParams[1]/Cd, 0]),
    ('Skylight', 'Flow Entering'): np.array([optParams[2]/Cd, 0]),
    ('Skylight', 'Flow Exiting'):  np.array([optParams[3]/Cd, 0])
    }

flowStatsMI[["C_d", "qRMS"]] = flowStatsMI["openingType"].apply(
    lambda ot: pd.Series(_params_for_opening(ot))
)

flowStatsMI["qRMS"] = flowStatsMI.apply(lambda x: [x["rms-mass_flux"], x["rms-mass_flux"]], axis=1)
flowStatsMI[["C_d", "qRMS"]]

In [ ]:
flowStatsMI.drop(columns=["pRMS"], inplace=True)
flowStatsMI_directNetwork4, roomVentilationMI_directNetwork4 = feUtils.update_flow_and_ventilation(flowStatsMI, roomVentilationMI, useDoors=True, tempStack=True, doorCd = [0.6, 0.6], checkAnalytical=False,
    pTypes = {
        "p-noInt-direct": "p_avg-noInt"
    }
)

In [ ]:
fig, axs, stats_direct_network4 = plot_direct_network_comparison(
    "Direct Network 4",
    flowStatsMI_directNetwork4,
    roomVentilationMI_directNetwork4,
)
